# Chapter 13: validation summary and the model error chart
The soft sensor from chapter 8, validated on the newest 75 fills of the 300 fill table. Then 44 live fills after go live: malt lot M-26-0188 arrives at live fill 25 and the beer finishes 0.12 degrees Plato higher than the model expects.

**No code tier.** Take the last 20 pairs of prediction and lab result. Subtract them and find the mean and sd. Draw the differences in order with lines at mean plus and minus 3 sd.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from scipy.stats import norm
from sklearn.linear_model import LinearRegression

In [ ]:
f = pd.read_csv(DATA + "ch08_fills.csv")
feats = ["ae_48h_plato", "og_plato", "mean_temp_c", "pitch_mcells_ml", "yeast_generation"]
train, test = f[f.split == "train"], f[f.split == "test"]
lin = LinearRegression().fit(train[feats], train.rdf_pct)
d = lin.predict(test[feats]) - test.rdf_pct.to_numpy(); bias, sd = d.mean(), d.std(ddof=1)
criteria = pd.DataFrame([("bias within 0.20 RDF points", round(bias, 2), abs(bias) <= 0.20), ("sd of differences 0.75 or less", round(sd, 2), sd <= 0.75)],
                        columns=["criterion", "result", "pass"])
b2, s2 = round(bias, 2), round(sd, 2)   # the book rounds bias and sd before the limits
print(f"bias {b2} sd {s2}; limits of agreement {b2 - 1.96 * s2:.2f} to {b2 + 1.96 * s2:.2f}; span {3.92 * s2:.2f} points = {3.92 * s2 / 6.7:.2f} degrees Plato")
print(f"standard error of the mean error with 75 fills: {sd / math.sqrt(75):.3f}")
criteria

In [ ]:
live = pd.read_csv(DATA + "ch13_live_fills.csv")
err = lin.predict(live[feats]) - live.rdf_pct.to_numpy()
cl, ucl, lcl = -0.06, -0.06 + 3 * 0.67, -0.06 - 3 * 0.67
change = live.index[live.malt_lot == "M-26-0188"][0]
plt.figure(figsize=(9, 3.5)); plt.plot(np.arange(1, len(err) + 1), err, "-o", ms=3, color="#2f7f72")
for y, st in ((cl, "-"), (ucl, "--"), (lcl, "--")): plt.axhline(y, ls=st, color="#0b4f45")
plt.axvline(change + 0.5, color="#c27c0e", ls=":", label="malt lot M-26-0188")
plt.xlabel("live fill"); plt.ylabel("model minus lab (RDF points)"); plt.legend(); plt.show()
run = 0
for i, e in enumerate(err, 1):
    run = run + 1 if e > cl else 0
    if run == 8: print("eight in a row above the centre at live fill", i); break

In [ ]:
shadow = err[change:change + 10]
offset = shadow.mean() - cl
print(f"mean error over the 10 shadow fills: {shadow.mean():.2f}; refit the offset by {offset:.2f}")
reval = err[change + 10:] - offset
print(f"revalidation bias on the next {len(reval)} fills: {reval.mean():.2f} (must sit within 0.20)")
z = (ucl - 0.74) / 0.67; p = norm.cdf(0.80 / 0.67)
print(f"single point beyond the limit: {1 - norm.cdf(z):.1%} a fill; eight in a row: {(1 - p ** 8) / ((1 - p) * p ** 8):.1f} fills on average")

**What to look at.** Before the malt change the error chart is quiet. After it the points sit above the centre and the run rule finds the shift long before a point crosses the limit. Version 1.0 becomes 1.1 only after the revalidation.